# L - Liskov Substitution (Sustitución de Liskov)
Ejemplo: Enemigos en un juego. Violación: una subclase que cambia contrato del padre; Corrección: respetar el contrato.


In [1]:
# --- Viola Liskov: Enemy.move en la subclase tiene firma distinta y lanza excepción ---
class Enemy:
    def __init__(self, name: str, hp: int):
        self.name = name
        self.hp = hp
    def move(self, dx: int, dy: int):
        # mueve por el mapa
        return (dx, dy)
    def attack(self, target):
        target.hp -= 5

class FlyingEnemy(Enemy):
    def __init__(self, name: str, hp: int, altitude: int = 10):
        super().__init__(name, hp)
        self.altitude = altitude
    def move(self, *args, **kwargs):
        # rompe la expectativa: recibe diferente firma y a veces no mueve en el mismo sentido
        raise RuntimeError('FlyingEnemy uses a different movement system')

def move_enemy_and_report(e: Enemy):
    # código cliente que espera poder llamar move(x,y)
    pos = e.move(1, 0)
    print('moved to', pos)

en = Enemy('Goblin', 30)
fe = FlyingEnemy('Wyvern', 50)
move_enemy_and_report(en)
# esto falla cuando se pasa fe -> rompe Liskov


moved to (1, 0)


In [2]:
# --- Versión corregida: definir abstracción común y mantener contrato ---
class EnemyBase:
    def __init__(self, name: str, hp: int):
        self.name = name
        self.hp = hp
    def move(self, x: int, y: int) -> tuple:
        raise NotImplementedError
    def attack(self, target):
        raise NotImplementedError

class GroundEnemy(EnemyBase):
    def __init__(self, name: str, hp: int, armor: int = 0):
        super().__init__(name, hp)
        self.armor = armor
    def move(self, x: int, y: int) -> tuple:
        return (x, y)
    def attack(self, target):
        target.hp -= max(1, 5 - self.armor)

class AirEnemy(EnemyBase):
    def __init__(self, name: str, hp: int, wingspan: int = 5):
        super().__init__(name, hp)
        self.wingspan = wingspan
    def move(self, x: int, y: int) -> tuple:
        # mantiene la misma firma y comportamiento observable (devuelve nueva posición)
        # implementa su detalle internamente (altitud), pero cumple el contrato
        return (x, y, self.wingspan)
    def attack(self, target):
        target.hp -= 7

def move_enemy_and_report_fixed(e: EnemyBase):
    pos = e.move(1, 0)
    print('moved to', pos)

g = GroundEnemy('Orc', 40, armor=1)
a = AirEnemy('Drake', 60, wingspan=8)
move_enemy_and_report_fixed(g)
move_enemy_and_report_fixed(a)
# Ambas subclases respetan el contrato de EnemyBase -> se pueden sustituir


moved to (1, 0)
moved to (1, 0, 8)
